# Attempt at Week2 Challenge
## Transforming Deep Research into an Agentic Flow
### We will attempt to create three top level agents: Preparer, Reporter, and Verifier
Perparer:

    * Generates 3 Questions for the topic
    * Generates 3 Search Strings for the topic


In [35]:
from agents import Agent, Runner, trace, AsyncOpenAI, OpenAIChatCompletionsModel, function_tool, WebSearchTool
from pydantic import BaseModel, Field
from typing import List
import asyncio

### Define the topic to research

In [2]:
topic = "Kobe Bryant"

## Let's define and test a Questioner Agent

In [ ]:
# define output types
class Questions(BaseModel):
    questions: List[str]
    """The questions to be asked and tested for"""

questioner_agent_instruction = f"You are given a topic and are tasked with generating a very diffiult question pertaining to that topic."
questioner_agent = Agent(name="questioner", instructions=questioner_agent_instruction, model="gpt-4.1-mini", output_type=Questions)

with trace("Questioner-Test"):
    result = await Runner.run(questioner_agent, topic)
    print(result.final_output)

questions=["Analyze the impact of Kobe Bryant's 'Mamba Mentality' on the evolution of professional basketball training methodologies and athlete mindset development.", "Discuss the influence of Kobe Bryant's international career, particularly his performance and leadership in the 2008 and 2012 Olympics, on the global perception of basketball and its growth outside the United States.", "Examine the complexity of Kobe Bryant's legacy in light of both his extraordinary achievements on the court and the controversies off the court, considering the broader implications for athlete role models in contemporary society."]


### Define the Search Planner Agent

In [ ]:
# define output types
class Search(BaseModel):
    search_string: str = Field("The suggested string to search for")
    reason: str = Field("Why you think this string is valuable to search for")

class SearchPlan(BaseModel):
    searches: List[Search]
    """A list of strings used to search the web for learning purposes"""
    
search_planner_instructions = f"You are a helpful research assistant. Given a topic, come up with a web search string;  Do not actually search the web, just tell me what you would search for on the web"
search_planner_agent = Agent(name="search_planner", instructions=search_planner_instructions, model="gpt-4.1-mini", output_type=SearchPlan)

# test it
with trace("Search-String-Test"):
    result = await Runner.run(search_planner_agent, topic)
    print(result.final_output)

searches=[Search(search_string='Kobe Bryant biography', reason="To get an overview of Kobe Bryant's life and career."), Search(search_string='Kobe Bryant career achievements and statistics', reason='To find detailed information on his basketball records and milestones.'), Search(search_string='Kobe Bryant tragic helicopter accident details', reason='To understand the circumstances and impact of his death.'), Search(search_string='Kobe Bryant legacy and influence on basketball', reason='To explore how he influenced the sport and other players.'), Search(search_string='Kobe Bryant famous quotes and interviews', reason='To learn about his personality, mindset, and philosophy from his own words.')]


### Create the WebSearch Agent
This agent will search the web on a given topic
Use Ollama for WebSearch to minimize costs (gpt charges more for using the WebSearchTool)

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
o_client = AsyncOpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")
o_model = OpenAIChatCompletionsModel(model="llama3.2", openai_client=o_client)

In [ ]:
@function_tool
def custom_websearch_tool():
    """Custom WebSearchTool"""
    return WebSearchTool(search_context_size="low")

In [23]:
web_search_agent_instruction = "You are researcher.  You are provided a topic and are required to use your websearch tool to gather information on the topic"
web_search_agent = Agent(name="search_agent", instructions=web_search_agent_instruction, tools=[custom_websearch_tool], model=o_model)

# test it
with trace("WebSearch-Test"):
    result = await Runner.run(web_search_agent, topic)
    print(result.final_output)

**Kobe Bryant: A Legendary Basketball Player and Entrepreneur**

Kobe Bryant (1978-2020) was an American professional basketball player, entrepreneur, and Academy Award-winning filmmaker. He is widely regarded as one of the greatest basketball players of all time.

**Early Life and Career**

Born in Philadelphia, Pennsylvania, Bryant began playing basketball at a young age. He joined the NBA directly from high school and was drafted by the Charlotte Hornets with the 13th overall pick in the 1996 NBA Draft. However, he was immediately traded to the Los Angeles Lakers on draft night.

**NBA Career**

Bryant spent his entire 20-year NBA career with the Los Angeles Lakers (1996-2016). He won numerous accolades, including:

* 5 NBA championships (2000-2002, 2009-2010)
* 2 NBA Finals MVP awards
* 1 regular season MVP award
* 18 All-Star Game appearances
* 11 All-NBA Team selections

**Personal Life and Legacy**

Bryant was married to Vanessa Laine Bryant, with whom he had four daughters. He 

### Create a Report Generator Agent
This agent will generate a report using the given information

In [ ]:
# define output types for the reporter
class Report(BaseModel):
    report: str = Field("The Actual Report")

report_generator_instructions="You are a well known journalist.  You are tasked with taking as input a some data and generating a long, detailed report with atleast 1000 words.  Make this formal, as if millions of people are going to read it."
report_generator_agent = Agent(name="report_generator", instructions=report_generator_instructions, model="gpt-4.1-mini", output_type=Report)

with trace("report-generator-test"):
    result = await Runner.run(report_generator_agent, result.final_output)
    print(result.final_output)

### Create top level Agents
Here, we create the Preparer and Report Agent

In [ ]:
# now, the report agent

# define function tool to explicitly orchestrate order of operations
@function_tool
async def reporter_tool(searches: List[str]) -> Report:
    """Report Generator Tool will execute Searches asychronously first, after which it will generate the actual report"""
    # run all searches asynchronously
    coroutines = [Runner.run(web_search_agent, search) for search in searches]
    result = await asyncio.gather(
        *coroutines,
    )

    search_results = '\n\n'.join([r.final_output for r in result])
    # generate the report
    result = await Runner.run(report_generator_agent, "generate a report using the details below: " + search_results)
    return result

# define the report agent
report_agent_instructions = "You are tasked with two things: researching the provided input strings and generating a report with your findings;  Use your tool, reporter_tool, to complete your task."
report_agent = Agent(name="report_generator", instructions=report_agent_instructions, model="gpt-4.1-mini", tools=[reporter_tool])

# some test data
test_search_strings = [
"Kobe Bryant biography",
"Kobe Bryant basketball achievements",
"Kobe Bryant tragic helicopter crash details",
"Kobe Bryant post-retirement activities",
"Kobe Bryant legacy and influence on basketball",
]

with trace("Report-Test"):
    result = await Runner.run(report_agent, " ".join(test_search_strings))
    print(result.final_output)



**Kobe Bryant: A Legacy of Excellence On and Off the Court**

Kobe Bean Bryant (August 23, 1978 – January 26, 2020) remains an indelible figure in basketball and beyond, celebrated as one of the most extraordinary athletes and multifaceted personalities of his generation. Over a two-decade career, he exhibited relentless dedication and incomparable performance on the court. After retiring, he ventured into filmmaking, entrepreneurship, and philanthropy, further cementing his global stature. His untimely death in a tragic helicopter crash shocked the world but could not diminish his lasting influence.

---

### Early Life and Foundation of a Champion

Born in Philadelphia, Pennsylvania, Kobe was introduced early to basketball through his father, Joe "Jellybean" Bryant, a former professional player. At Lower Merion High School, he excelled in multiple sports but particularly stood out in basketball, earning the National Player of the Year in 1996.

---

### The Dawn of an NBA Legend: Pro

In [ ]:
NUM_QUESTIONS = 3
NUM_SEARCH_STRS = 5

preparer_agent_instructions = f"""
You are an assistant preparing your worker to perform some deep research on a topic.
You are tasked with coming up with {NUM_QUESTIONS} very difficult questions related to this topic. 
In addition, you are also tasked with preparing {NUM_SEARCH_STRS} search strings for your worker to use during their research
After you've received all questions and search strings, handoff the data to the reporter agent.  
"""
preparer_agent_tools=[
    questioner_agent.as_tool(tool_name="question_tool", tool_description="this tool is used to generated questions related to a given topic"),
    search_planner_agent.as_tool(tool_name="search_planner_tool", tool_description="this tool is used to generated searching strings for a given topic"),
]
preparer_agent = Agent(name="preparer",
                       instructions=preparer_agent_instructions,
                       model="gpt-4.1-mini",
                       tools=preparer_agent_tools,
                       handoffs=[report_agent],
                       )

# run it !! 
with trace("preparer-agent-test"):
    result = await Runner.run(preparer_agent, topic)
    print(result.final_output)

Here are 3 very difficult questions related to Kobe Bryant for deep research:

1. Analyze the impact of Kobe Bryant's work ethic and mentality, often termed the 'Mamba Mentality,' on the development of basketball training regimens in the NBA and collegiate systems.
2. Evaluate the statistical significance of Kobe Bryant's scoring achievements relative to his era's pace of play and defensive strategies, using advanced metrics like Player Efficiency Rating (PER) and Win Shares.
3. Discuss the evolution of Kobe Bryant's playing style over his 20-year career, focusing on how changes in his physical condition and team composition influenced his offensive and defensive roles on the court.

And here are 5 search strings to assist in your research:

1. Kobe Bryant biography
2. Kobe Bryant basketball achievements
3. Kobe Bryant tragic helicopter crash details
4. Kobe Bryant post-retirement activities
5. Kobe Bryant legacy and influence on basketball
